# Assignment 1: Data Loading, Storage and File Formats

Load the Superstore sales data from a public CSV, save a local copy, clean it, then analyse sales by region, category and month.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
URL = 'https://raw.githubusercontent.com/Technology-Hatchery/Rcode/master/Data/Sample%20-%20Superstore%20Sales%20(Excel).csv'
raw = pd.read_csv(URL, encoding='latin1')
raw.to_csv(DATA_DIR / 'superstore_sales.csv', index=False)
print('Loaded', len(raw), 'rows; saved a copy to', DATA_DIR / 'superstore_sales.csv')
raw.head()

## Inspect the data

In [ ]:
print('Shape:', raw.shape)
display(raw.dtypes.to_frame('data_type'))
display(raw.describe(include='all').T)
print('Missing values:')
display(raw.isna().sum().to_frame('count'))
print('Duplicate rows:', raw.duplicated().sum())

## Clean and prepare

Remove duplicate rows, parse dates, convert numeric columns, and discard rows missing key sales fields.

In [ ]:
sales = raw.copy()
sales.columns = sales.columns.str.strip()
sales = sales.drop_duplicates()
rename = {'Order Date':'OrderDate', 'Product Name':'Product', 'Sub-Category':'SubCategory'}
sales = sales.rename(columns={old: new for old, new in rename.items() if old in sales.columns})
if 'OrderDate' in sales.columns:
    sales['OrderDate'] = pd.to_datetime(sales['OrderDate'], errors='coerce')
for col in ['Sales', 'Quantity', 'Profit', 'Discount']:
    if col in sales.columns:
        sales[col] = pd.to_numeric(sales[col], errors='coerce')
key_cols = [c for c in ['OrderDate', 'Region', 'Category', 'Sales', 'Quantity'] if c in sales.columns]
sales = sales.dropna(subset=key_cols)
if {'Sales', 'Quantity'}.issubset(sales.columns):
    sales = sales[(sales['Sales'] >= 0) & (sales['Quantity'] > 0)].copy()
if 'OrderDate' in sales.columns:
    sales['Month'] = sales['OrderDate'].dt.to_period('M').astype(str)
print('Rows after cleaning:', len(sales))
sales.head()

## Analyze sales

Calculate descriptive statistics and aggregate results by region, category, and month.

In [ ]:
print(sales.columns.tolist())

In [ ]:
display(sales[['Sales', 'Order Quantity', 'Profit', 'Discount']].describe().T)

print('Total sales:', round(sales['Sales'].sum(), 2))

print('Average order-line sales:', round(sales['Sales'].mean(), 2))

by_region = sales.groupby('Region', as_index=False).agg(
    Sales=('Sales', 'sum'),
    Profit=('Profit', 'sum')
)

by_category = sales.groupby('Product Category', as_index=False).agg(
    Sales=('Sales', 'sum'),
    Profit=('Profit', 'sum')
)

display(by_region.sort_values('Sales', ascending=False))

display(by_category.sort_values('Sales', ascending=False))

by_month = sales.groupby('Month', as_index=False)['Sales'].sum()

display(by_month.head())

## Visualize the results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

by_region.plot.bar(
    x='Region',
    y='Sales',
    ax=axes[0],
    legend=False,
    title='Sales by Region',
    color='steelblue'
)

by_category.plot.bar(
    x='Product Category',
    y='Sales',
    ax=axes[1],
    legend=False,
    title='Sales by Category',
    color='seagreen'
)

by_month.plot(
    x='Month',
    y='Sales',
    ax=axes[2],
    legend=False,
    title='Monthly Sales',
    marker='o',
    color='darkorange'
)

for ax in axes:
    ax.set_ylabel('Sales')
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

sales.boxplot(
    column='Sales',
    by='Product Category',
    figsize=(7, 4)
)

plt.title('Sales Distribution by Category')
plt.suptitle('')
plt.show()

## Save the cleaned data

In [ ]:
output = DATA_DIR / 'superstore_sales_cleaned.csv'
sales.to_csv(output, index=False)
print('Saved cleaned data to', output.resolve())